In [8]:
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import cross_val_score, StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [9]:
# Загружаем предобработанные данные из artifacts
X_train = np.load('../artifacts/X_train_preprocessed.npy')
X_test = np.load('../artifacts/X_test_preprocessed.npy')
y_train = np.load('../artifacts/y_train.npy')
y_test = np.load('../artifacts/y_test.npy')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Доля оттока в train: {y_train.mean():.2%}")
print(f"Доля оттока в test: {y_test.mean():.2%}")

X_train shape: (5634, 38)
X_test shape: (1409, 38)
Доля оттока в train: 26.54%
Доля оттока в test: 26.54%


In [10]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': model_name,
        'AUC-ROC': roc_auc_score(y_test, y_pred_proba),
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)
    }
    return metrics, model

In [13]:
lr = LogisticRegression(max_iter=1000, random_state=42)
metrics_lr, lr_model = evaluate_model(lr, X_train, y_train, X_test, y_test, "Logistic Regression")
print("Логистическая регрессия:")
print(classification_report(y_test, lr_model.predict(X_test)))

Логистическая регрессия:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [14]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
metrics_rf, rf_model = evaluate_model(rf, X_train, y_train, X_test, y_test, "Random Forest")
print("Случайный лес:")
print(classification_report(y_test, rf_model.predict(X_test)))

Случайный лес:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [15]:
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
metrics_xgb, xgb_model = evaluate_model(xgb, X_train, y_train, X_test, y_test, "XGBoost")
print("XGBoost:")
print(classification_report(y_test, xgb_model.predict(X_test)))

XGBoost:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.62      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [16]:
results = pd.DataFrame([metrics_lr, metrics_rf, metrics_xgb])
results = results.round(4)
print("Сравнение моделей на тестовой выборке:")
print(results.to_string(index=False))

Сравнение моделей на тестовой выборке:
              Model  AUC-ROC     F1  Precision  Recall
Logistic Regression   0.8428 0.5872     0.6633  0.5267
      Random Forest   0.8210 0.5625     0.6342  0.5053
            XGBoost   0.8425 0.5808     0.6246  0.5428


In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"Логистическая регрессия – кросс-валидация (AUC-ROC): {cv_scores}")
print(f"Среднее: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

Логистическая регрессия – кросс-валидация (AUC-ROC): [0.84811287 0.83003934 0.84248623 0.85940454 0.85887258]
Среднее: 0.8478 (+/- 0.0110)


In [21]:
# Определяем лучшую модель по AUC-ROC
results_df = pd.DataFrame([metrics_lr, metrics_rf, metrics_xgb])
best_model_name = results_df.loc[results_df['AUC-ROC'].idxmax(), 'Model']
print(f"Лучшая базовая модель по AUC-ROC: {best_model_name}")

if best_model_name == "Logistic Regression":
    best_baseline_model = lr_model
elif best_model_name == "Random Forest":
    best_baseline_model = rf_model
else:
    best_baseline_model = xgb_model

# Сохраняем лучшую модель
joblib.dump(best_baseline_model, '../artifacts/baseline_best_model.joblib')
print(f"Базовая модель {best_model_name} сохранена в artifacts/baseline_best_model.joblib")

Лучшая базовая модель по AUC-ROC: Logistic Regression
Базовая модель Logistic Regression сохранена в artifacts/baseline_best_model.joblib
